# Reference Trainer Baseline — ResNet-20 on CIFAR-10

## Purpose
This notebook establishes an upper-bound synchronous baseline by faithfully replicating the training recipe from the reference implementation by **Yerlan Idelbayev**:

> Idelbayev, Y. *Proper ResNets for CIFAR-10 in PyTorch*. GitHub: [akamaster/pytorch_resnet_cifar10](https://github.com/akamaster/pytorch_resnet_cifar10)

The reference reports **~91.25% top-1 accuracy** for ResNet-20 on CIFAR-10.

## How this differs from `train_baseline.ipynb`
| Aspect | `train_baseline.ipynb` | This notebook |
|---|---|---|
| Momentum | 0.0 (vanilla SGD) | **0.9** |
| Epochs | 164 | **200** |
| LR milestones | [82, 123] | **[100, 150]** |
| Normalization | CIFAR-10 stats | **CIFAR-10 stats** |
| Per-batch metrics | No | **Yes** (AverageMeter) |

The momentum and learning-rate schedule are the primary drivers of the improved accuracy.

In [ ]:
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score

sys.path.append(str(Path.cwd().parent))
from src.model import resnet20

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cudnn.benchmark = True
device

device(type='cuda')

In [ ]:
# CIFAR-10 normalization statistics
normalize = transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                                 std=[0.2023, 0.1994, 0.2010])

DATA_ROOT = "../../data"

train_loader = torch.utils.data.DataLoader(
    datasets.CIFAR10(
        root=DATA_ROOT, train=True, download=True,
        transform=transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            normalize,
        ])
    ),
    batch_size=128, shuffle=True, num_workers=4, pin_memory=True
)

val_loader = torch.utils.data.DataLoader(
    datasets.CIFAR10(
        root=DATA_ROOT, train=False, download=False,
        transform=transforms.Compose([
            transforms.ToTensor(),
            normalize,
        ])
    ),
    batch_size=128, shuffle=False, num_workers=4, pin_memory=True
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [5]:
model = resnet20().to(device)

criterion = nn.CrossEntropyLoss().to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[100, 150],
    gamma=0.1
)

In [ ]:
PRINT_FREQ = 50
NUM_EPOCHS = 200
best_prec1 = 0.0

for epoch in tqdm(range(NUM_EPOCHS), desc="Epochs"):
    # --- Train ---
    model.train()
    running_loss = 0.0
    n_samples = 0
    train_preds, train_labels = [], []
    batch_times = []

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  lr={optimizer.param_groups[0]['lr']:.5e}")
    end = time.time()

    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = outputs.detach().max(1)
        train_preds.extend(predicted.cpu().numpy())
        train_labels.extend(targets.cpu().numpy())
        running_loss += loss.item() * inputs.size(0)
        n_samples += inputs.size(0)
        batch_times.append(time.time() - end)
        end = time.time()

        if i % PRINT_FREQ == 0:
            avg_t = sum(batch_times) / len(batch_times)
            print(f"  [{i}/{len(train_loader)}]  "
                  f"Time {batch_times[-1]:.3f} ({avg_t:.3f})  "
                  f"Loss {loss.item():.4f} ({running_loss / n_samples:.4f})")

    scheduler.step()

    train_acc = accuracy_score(train_labels, train_preds) * 100
    train_f1  = f1_score(train_labels, train_preds, average="macro") * 100
    avg_train_loss = running_loss / n_samples

    # --- Validate ---
    model.eval()
    val_running_loss = 0.0
    val_n = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            _, predicted = outputs.max(1)
            val_preds.extend(predicted.cpu().numpy())
            val_labels.extend(targets.cpu().numpy())
            val_running_loss += loss.item() * inputs.size(0)
            val_n += inputs.size(0)

    val_acc = accuracy_score(val_labels, val_preds) * 100
    val_f1  = f1_score(val_labels, val_preds, average="macro") * 100
    avg_val_loss = val_running_loss / val_n

    best_prec1 = max(best_prec1, val_acc)
    print(f"  Train Loss: {avg_train_loss:.4f}  Train Acc: {train_acc:.2f}%  Train Macro F1: {train_f1:.2f}%")
    print(f"  Val   Loss: {avg_val_loss:.4f}  Val   Acc: {val_acc:.2f}%  Val   Macro F1: {val_f1:.2f}%  Best: {best_prec1:.2f}%")

In [7]:
print(f"Training complete.")
print(f"Best Prec@1: {best_prec1:.2f}%")

Training complete.
Best Prec@1: 91.72%


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

all_preds  = np.array(val_preds)
all_labels = np.array(val_labels)

acc      = accuracy_score(all_labels, all_preds) * 100
macro_f1 = f1_score(all_labels, all_preds, average="macro") * 100

print(f"Accuracy : {acc:.2f}%")
print(f"Macro F1 : {macro_f1:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
fig.colorbar(im, ax=ax)
ax.set(
    xticks=range(len(CIFAR10_CLASSES)),
    yticks=range(len(CIFAR10_CLASSES)),
    xticklabels=CIFAR10_CLASSES,
    yticklabels=CIFAR10_CLASSES,
    xlabel="Predicted",
    ylabel="True",
    title="Confusion Matrix — ResNet-20 on CIFAR-10 (Reference Baseline)",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
for i in range(len(CIFAR10_CLASSES)):
    for j in range(len(CIFAR10_CLASSES)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.tight_layout()
plt.show()